<a href="https://colab.research.google.com/github/ANKIT-KANDULNA/CS318_DL-LAB/blob/main/Experiment-4/DL_lab_exp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
from collections import Counter

In [21]:
!pip install gdown
!gdown 1ko4cu8nWhtoQ3OoTqkJdesoq3GtiToev

Downloading...
From: https://drive.google.com/uc?id=1ko4cu8nWhtoQ3OoTqkJdesoq3GtiToev
To: /content/poems-100.csv
100% 140k/140k [00:00<00:00, 73.7MB/s]


In [22]:
with open("poems-100.csv","r",encoding="utf-8") as f:
    text = f.read().lower()

print(text[:500])

text
"o my luve's like a red, red rose
that’s newly sprung in june;
o my luve's like the melodie
that’s sweetly play'd in tune.

as fair art thou, my bonnie lass,
so deep in luve am i:
and i will luve thee still, my dear,
till a’ the seas gang dry:

till a’ the seas gang dry, my dear,
and the rocks melt wi’ the sun:
i will luve thee still, my dear,
while the sands o’ life shall run.

and fare thee well, my only luve
and fare thee well, a while!
and i will come again, my luve,
tho’ it were ten th


# Tokenization

In [30]:
# Keep only top 2500 words
word_counts = Counter(words)
most_common = word_counts.most_common(2500)

vocab = [w for w,_ in most_common]
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}
vocab_size = len(vocab)

# Filter words not in vocab
filtered_words = [w for w in words if w in word2idx]
print(f"Vocab size:", vocab_size)

Vocab size: 2500


In [24]:
seq_length = 3

inputs = []
targets = []

for i in range(len(filtered_words)-seq_length):
    input_seq = filtered_words[i:i+seq_length]
    target_word = filtered_words[i+seq_length]

    inputs.append([word2idx[w] for w in input_seq])
    targets.append(word2idx[target_word])


# ONE-HOT ENCODING

In [25]:
class OneHotDataset(Dataset):
    def __init__(self,inputs,targets,vocab_size):
        self.inputs = inputs
        self.targets = targets
        self.vocab_size = vocab_size

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self,idx):
        x = torch.zeros(seq_length,self.vocab_size)
        for i,word_idx in enumerate(self.inputs[idx]):
            x[i][word_idx] = 1.0
        y = torch.tensor(self.targets[idx])
        return x,y


In [26]:
dataset1 = OneHotDataset(inputs,targets,vocab_size)
loader1 = DataLoader(dataset1,batch_size=32,shuffle=True)

RNN Model (One-hot)

In [27]:
class RNN_OneHot(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size,hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,output_size)

    def forward(self,x):
        out,_ = self.rnn(x)
        out = self.fc(out[:,-1,:])
        return out


LTSM Model (One-hot)

In [28]:
class LSTM_OneHot(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size,hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,output_size)

    def forward(self,x):
        out,_ = self.lstm(x)
        out = self.fc(out[:,-1,:])
        return out


# EMBEDDING APPROACH

Dataset (Indexed Only)

In [31]:
class IndexDataset(Dataset):
    def __init__(self,inputs,targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self,idx):
        return torch.tensor(self.inputs[idx]), torch.tensor(self.targets[idx])


In [32]:
dataset2 = IndexDataset(inputs,targets)
loader2 = DataLoader(dataset2,batch_size=32,shuffle=True)

RNN with Embedding

In [33]:
class RNN_Embedding(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_size)
        self.rnn = nn.RNN(embed_size,hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,vocab_size)

    def forward(self,x):
        x = self.embedding(x)
        out,_ = self.rnn(x)
        out = self.fc(out[:,-1,:])
        return out


LSTM with Embedding

In [34]:
class LSTM_Embedding(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_size)
        self.lstm = nn.LSTM(embed_size,hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,vocab_size)

    def forward(self,x):
        x = self.embedding(x)
        out,_ = self.lstm(x)
        out = self.fc(out[:,-1,:])
        return out


# Training Function

In [35]:
def train_model(model,loader,epochs=2):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(),lr=0.001)

    for epoch in range(epochs):
        total_loss = 0
        for x,y in loader:
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output,y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss:{total_loss/len(loader)}")


In [36]:
hidden_size = 64
embed_size = 50

# OneHot RNN
model_rnn_onehot = RNN_OneHot(vocab_size,hidden_size,vocab_size)
train_model(model_rnn_onehot,loader1,epochs=2)

# OneHot LSTM
model_lstm_onehot = LSTM_OneHot(vocab_size,hidden_size,vocab_size)
train_model(model_lstm_onehot,loader1,epochs=2)

# Embedding RNN
model_rnn_embed = RNN_Embedding(vocab_size,embed_size,hidden_size)
train_model(model_rnn_embed,loader2,epochs=2)

# Embedding LSTM
model_lstm_embed = LSTM_Embedding(vocab_size,embed_size,hidden_size)
train_model(model_lstm_embed,loader2,epochs=2)

Epoch 1, Loss:6.246636397465711
Epoch 2, Loss:5.84878271337934
Epoch 1, Loss:6.306899494846097
Epoch 2, Loss:5.913042839104531
Epoch 1, Loss:6.274195117408065
Epoch 2, Loss:5.7028414461096695
Epoch 1, Loss:6.315904890003174
Epoch 2, Loss:5.785554950647821


# Text Generation

In [38]:
def generate_text(model,start_words,num_words=15):
    model.eval()
    words_input = start_words.split()

    for _ in range(num_words):
        input_idx = [word2idx[w] for w in words_input[-seq_length:]]
        x = torch.tensor([input_idx])

        if isinstance(model,RNN_OneHot) or isinstance(model,LSTM_OneHot):
            onehot = torch.zeros(1,seq_length,vocab_size)
            for i,idx in enumerate(input_idx):
                onehot[0,i,idx]=1
            output = model(onehot)
        else:
            output = model(x)

        predicted = torch.argmax(output,dim=1).item()
        words_input.append(idx2word[predicted])

    return " ".join(words_input)

In [39]:
print(generate_text(model_lstm_embed,"i love"))

i love and the of the of the of the of the of the of the of
